<a href="https://colab.research.google.com/github/climatescience/Spatial-data-visualization-with-Python/blob/main/4_panel_correlation_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# --- CONFIGURE FILE NAMES ---
net_rad_era_path = 'localcor_ERA_MODIS_RN.tif'     # Top-Left
net_rad_modis_path = 'localcor_ERA_MODIS_LE.tif' # Top-Right
le_era_path = 'localcor_ERA_MODIS_SH.tif'               # Bottom-Left
le_modis_path = 'localcor_ERA_Bulk_SH.tif'           # Bottom-Right

shapefile_path = 'PAK_adm0.shp'

# Helper function to read GeoTIFF, handle NoData, and get spatial extent
def read_tif(path):
    with rasterio.open(path) as src:
        data = src.read(1)
        nodata = src.nodata
        if nodata is not None:
            data = np.where(data == nodata, np.nan, data)
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    return data, extent

# Load the 4 datasets and their respective extents
nr_era, ext_nr_era = read_tif(net_rad_era_path)
nr_modis, ext_nr_mod = read_tif(net_rad_modis_path)
le_era, ext_le_era = read_tif(le_era_path)
le_modis, ext_le_mod = read_tif(le_modis_path)

# Load boundary shapefile and reproject to match raster CRS
with rasterio.open(net_rad_era_path) as src:
    raster_crs = src.crs

boundary_gdf = gpd.read_file(shapefile_path).to_crs(raster_crs)
minx, miny, maxx, maxy = boundary_gdf.total_bounds

# --- OPTIONAL: Define a shared global min and max for all datasets ---
# (You can also manually set these to fixed numbers, e.g., vmin=-100, vmax=100)
all_data = np.concatenate([nr_era.flatten(), nr_modis.flatten(), le_era.flatten(), le_modis.flatten()])
vmin = np.nanmin(all_data)
vmax = np.nanmax(all_data)

# 3. Create the 4-Panel Plot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.subplots_adjust(right=0.88, top=0.92, bottom=0.08, left=0.08, wspace=0.1, hspace=0.1)
datasets = [
    (nr_era, ext_nr_era,  'Net Radiation : MODIS'),
    (nr_modis, ext_nr_mod,'Latent Heat   : MODIS'),
    (le_era, ext_le_era,  'Sensible Heat : MODIS'),
    (le_modis, ext_le_mod, 'Sensible Heat: Bulk')
]

im = None
for ax, (data, ext, title) in zip(axes.flat, datasets):
    im = ax.imshow(data, cmap='jet', extent=ext, vmin=vmin, vmax=vmax)
    boundary_gdf.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.8)

    # Force all panels to share the same shapefile bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

# --- Add a Single Shared Colorbar on the RIGHT Side ---
# [left, bottom, width, height] in figure coordinates
cbar_ax = fig.add_axes([0.91, 0.15, 0.025, 0.7])
fig.colorbar(im, cax=cbar_ax, orientation='vertical', label='Local Pearson r')
cbar_ax.yaxis.set_ticks_position('right')
cbar_ax.yaxis.set_label_position('right')

plt.show()